# External-baseline comparison — TFDV / Deequ / Great Expectations

**NEXT_EXPERIMENTS item 6.** Runs each baseline's *default* checks on the **same degraded sources the gate sees** (regenerated by `experiments/baselines/gen_baseline_sources.py`, which reuses the frozen gate's `split_sources`/`degrade` with identical seeds). Records **per-mode detection recall** and the **clean false-flag rate**, aligned with the gate's `det_recall` in `crossdomain_<domain>.csv`.

**Framing (per the paper):** the contribution is *not* raw check novelty — it is the DP-private + cross-source + readiness-**decision** layer. This notebook makes that precise by showing what off-the-shelf validators do and do not catch per mode.

**DATA SAFETY.** This notebook defaults to **synthetic** data — nothing sensitive leaves your machine. `healthcare` must stay synthetic here (MIMIC-IV is DUA-bound; never upload its rows). `finance` (EDGAR) and `government` (CMS Open Payments) are public and may be pointed at an uploaded extract via `DATA_ROOT`.

> ⚠️ **These baseline cells are not pre-executed** (TFDV/Deequ do not run on the author's Windows box). Expect minor version/API fixups on first run — GE and Deequ APIs drift between releases. The GE version is pinned below for stability.

In [ ]:
# 1. Installs. IMPORTANT: TFDV and Deequ need Python <= 3.11. Colab defaults to 3.13
#    and Kaggle to 3.12 — where TFDV has NO wheel and pydeequ misbehaves. The reliable
#    path is the headless Docker runner (see experiments/baselines/run_baselines.py and
#    docs/NEXT_EXPERIMENTS.md item 6). If you stay here, GE works everywhere; Deequ needs
#    Python <=3.11 AND Spark 3.5/3.3/3.2 (NOT 3.4 — pydeequ rejects it); TFDV needs 3.11.
import sys, platform
print("Python:", platform.python_version(), "  (need 3.11.x for TFDV + Deequ)")

!pip install "great_expectations==0.18.19"
!pip install pydeequ "pyspark==3.5.1"
!pip install "tensorflow-data-validation==1.15.1"   # 3.11 wheel; unavailable on 3.12/3.13
import os
os.environ["SPARK_VERSION"] = "3.5"   # pydeequ supports 3.5/3.3/3.2, NOT 3.4

# Report exactly what imported — the grid (cell 7) SKIPS any baseline that didn't.
for m in ["great_expectations", "tensorflow_data_validation", "pydeequ"]:
    try:
        __import__(m); print("OK      ", m)
    except Exception as e:
        print("MISSING ", m, "->", type(e).__name__, str(e)[:80])
print("\nIf TFDV/pydeequ show MISSING and Python above is 3.12/3.13: use the Docker runner\n"
      "(docs/NEXT_EXPERIMENTS.md item 6) — it runs all of GE+Deequ (and TFDV if it resolves)\n"
      "in a Python 3.11 container. GE alone is a valid baseline; the grid runs any subset.")

In [ ]:
# 2. Locate the repo + pick a WRITABLE output dir. Works on Colab AND Kaggle:
#    - Colab: upload the repo zip (Files pane) -> auto-unzipped here.
#    - Kaggle: Add Input -> Datasets -> Upload highstakes-gate.zip (Kaggle auto-extracts
#      it, read-only, under /kaggle/input/<slug>/). This cell finds it automatically.
import sys, subprocess, pathlib, glob, zipfile

def _has_src(p):
    return pathlib.Path(p, "src").exists()

REPO_URL = ""          # optional Colab git clone
REPO_DIR = None

# Kaggle: search the read-only dataset mounts
for base in glob.glob("/kaggle/input/*"):
    if _has_src(base):
        REPO_DIR = base; break
    inner = [p for p in glob.glob(base + "/*") if _has_src(p)]
    if inner:
        REPO_DIR = inner[0]; break

# Colab / local: clone, or auto-unzip the newest uploaded zip
if REPO_DIR is None:
    REPO_DIR = "/content/highstakes-gate"
    if REPO_URL and not _has_src(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    elif not _has_src(REPO_DIR):
        zips = sorted(glob.glob("/content/*.zip"))
        assert zips, "Upload the repo zip (Colab Files pane) or add it as a Kaggle dataset."
        with zipfile.ZipFile(zips[-1]) as z:
            z.extractall(REPO_DIR)
        if not _has_src(REPO_DIR):
            innr = [str(p) for p in pathlib.Path(REPO_DIR).glob("*") if _has_src(p)]
            if innr:
                REPO_DIR = innr[0]

assert _has_src(REPO_DIR), f"{REPO_DIR} must contain src/ and experiments/."

# Kaggle's /kaggle/input is READ-ONLY -> write outputs to /kaggle/working instead.
OUT = "/kaggle/working" if pathlib.Path("/kaggle/working").exists() \
    else str(pathlib.Path(REPO_DIR, "experiments", "results"))

sys.path.insert(0, str(pathlib.Path(REPO_DIR, "src")))
sys.path.insert(0, str(pathlib.Path(REPO_DIR, "experiments", "baselines")))
from gen_baseline_sources import make_cell_sources
from domains import degradations as deg
print("repo:", REPO_DIR, "\noutput ->", OUT, "\nmodes:", deg.MODES)

In [ ]:
# 3. Grid config. Synthetic by default (safe). To use a PUBLIC extract for a domain,
#    add it to DATA_ROOT, e.g. DATA_ROOT = {"finance": "/content/edgar"}. Never healthcare.
DOMAINS     = ["healthcare", "finance", "government"]
MODES       = ["none"] + deg.MODES        # none = clean-only cell -> clean false-flag rate
SEVERITIES  = ["high"]                     # widen to deg.SEVERITIES for a full sweep
SEEDS       = list(range(5))
N_SOURCES, N_DEGRADED = 6, 2
N_ROWS, MAX_ROWS = 4000, 40000
DATA_ROOT   = {}                          # {} = all synthetic

def numeric_bounds(ref, col, lo_q=0.001, hi_q=0.999):
    import pandas as pd
    s = pd.to_numeric(ref[col], errors="coerce").dropna()
    return (float(s.quantile(lo_q)), float(s.quantile(hi_q)), float(s.mean())) if len(s) else (None, None, None)

In [ ]:
# 4. Great Expectations adapter (legacy 0.18 PandasDataset). GUARDED import: if GE
#    didn't install, HAVE_GE=False and ge_flag returns None so the grid skips it.
try:
    import great_expectations as ge
    HAVE_GE = True
except Exception as e:
    HAVE_GE = False
    print("GE unavailable ->", type(e).__name__, str(e)[:100])
import pandas as pd

def ge_flag(source_df, reference_df, domain):
    if not HAVE_GE:
        return None
    d = ge.from_pandas(source_df)
    ok = True
    for col in domain.required_columns:
        ok &= d.expect_column_to_exist(col).success
    for col in domain.feature_columns:
        if col in source_df.columns:
            ok &= d.expect_column_values_to_not_be_null(
                col, mostly=1.0 - domain.null_threshold).success
            lo, hi, _ = numeric_bounds(reference_df, col)
            if lo is not None and hi > lo:
                ok &= d.expect_column_values_to_be_between(
                    col, min_value=lo, max_value=hi, mostly=0.99).success
    lab = domain.label_column
    if lab in source_df.columns and lab in reference_df.columns:
        _, _, m = numeric_bounds(reference_df, lab)
        if m is not None:
            ok &= d.expect_column_mean_to_be_between(
                lab, min_value=0.7 * m, max_value=1.3 * m + 1e-9).success
    return not ok
print("GE adapter ready" if HAVE_GE else "GE adapter DISABLED (import failed)")

In [ ]:
# 5. TFDV adapter. GUARDED import. DEFAULT TFDV catches structural + presence; we add
#    a JSD drift_comparator so it can also flag distributional. If TFDV didn't install
#    (needs a runtime restart after pip), HAVE_TFDV=False and this baseline is skipped.
try:
    import tensorflow_data_validation as tfdv
    HAVE_TFDV = True
except Exception as e:
    HAVE_TFDV = False
    print("TFDV unavailable ->", type(e).__name__, str(e)[:100])

def tfdv_schema(reference_df, domain, drift_threshold=0.1):
    if not HAVE_TFDV:
        return None, None
    ref_stats = tfdv.generate_statistics_from_dataframe(reference_df)
    schema = tfdv.infer_schema(ref_stats)
    for col in domain.feature_columns:
        try:
            f = tfdv.get_feature(schema, col)
            f.drift_comparator.jensen_shannon_divergence.threshold = drift_threshold
        except Exception:
            pass
    return schema, ref_stats

def tfdv_flag(source_df, reference_df, domain, schema, ref_stats):
    if not HAVE_TFDV or schema is None:
        return None
    stats = tfdv.generate_statistics_from_dataframe(source_df)
    anomalies = tfdv.validate_statistics(stats, schema, previous_statistics=ref_stats)
    return len(anomalies.anomaly_info) > 0
print("TFDV adapter ready" if HAVE_TFDV else "TFDV adapter DISABLED (import failed)")

In [ ]:
# 6. Deequ adapter (pydeequ + Spark). GUARDED. On Colab Python 3.13, importing pydeequ
#    fails with "No module named 'typing.io'" (typing.io was REMOVED in 3.13) — we shim
#    it below so the import can proceed. NOTE: pyspark 3.4 officially supports Python
#    <=3.11, so on 3.13 the Spark session may still misbehave; the reliable fix is the
#    Python 3.11 fallback runtime (see cell 1). This baseline is skipped if it can't load.
import sys, types, typing
for _sub, _names in (("typing.io", ("IO", "TextIO", "BinaryIO")),
                     ("typing.re", ("Pattern", "Match"))):
    if _sub not in sys.modules:
        _m = types.ModuleType(_sub)
        for _n in _names:
            try:
                setattr(_m, _n, getattr(typing, _n))
            except AttributeError:
                pass
        sys.modules[_sub] = _m

try:
    import pydeequ
    from pyspark.sql import SparkSession
    from pydeequ.checks import Check, CheckLevel
    from pydeequ.verification import VerificationSuite, VerificationResult
    spark = (SparkSession.builder
             .config("spark.jars.packages", pydeequ.deequ_maven_coord)
             .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
             .getOrCreate())
    HAVE_DEEQU = True
except Exception as e:
    HAVE_DEEQU = False
    print("Deequ unavailable ->", type(e).__name__, str(e)[:120])

def deequ_flag(source_df, reference_df, domain):
    if not HAVE_DEEQU:
        return None
    # structural: a required column dropped entirely
    missing = [c for c in domain.required_columns if c not in source_df.columns]
    if missing:
        return True
    sdf = spark.createDataFrame(source_df)
    chk = Check(spark, CheckLevel.Error, "readiness")
    for col in domain.feature_columns:
        if col in source_df.columns:
            chk = chk.isComplete(col)
            lo, hi, _ = numeric_bounds(reference_df, col)
            if lo is not None and hi > lo:
                chk = chk.hasMean(col, lambda m, lo=lo, hi=hi: lo <= m <= hi)
    res = VerificationSuite(spark).onData(sdf).addCheck(chk).run()
    rdf = VerificationResult.checkResultsAsDataFrame(spark, res)
    statuses = [r["constraint_status"] for r in rdf.collect()]
    return any(s != "Success" for s in statuses)
print("Deequ adapter ready" if HAVE_DEEQU else "Deequ adapter DISABLED (import failed)")

In [ ]:
# 7. Run the grid. For each (domain, mode, severity, seed): regenerate the gate's
#    sources, run each baseline per source, aggregate per-mode detection recall
#    (over degraded sources) and clean false-flag rate (over clean sources).
import pandas as pd, math, numpy as np

ADAPTERS = {"great_expectations": "ge", "tfdv": "tfdv", "deequ": "deequ"}
rows = []
for domain_name in DOMAINS:
    for mode in MODES:
        sev_list = ["none"] if mode == "none" else SEVERITIES
        for severity in sev_list:
            for seed in SEEDS:
                cell = make_cell_sources(domain_name, mode, severity, seed,
                                         n_sources=N_SOURCES, n_degraded=N_DEGRADED,
                                         n_rows=N_ROWS, max_rows=MAX_ROWS,
                                         data_root=DATA_ROOT.get(domain_name))
                dom, ref, srcs = cell["domain"], cell["reference"], cell["sources"]
                schema = ref_stats = None
                per = {b: {"tp": 0, "deg": 0, "cf": 0, "cl": 0} for b in ADAPTERS}
                for src, is_deg in srcs:
                    for bname in ADAPTERS:
                        try:
                            if bname == "great_expectations":
                                fl = ge_flag(src, ref, dom)
                            elif bname == "tfdv":
                                if schema is None:
                                    schema, ref_stats = tfdv_schema(ref, dom)
                                fl = tfdv_flag(src, ref, dom, schema, ref_stats)
                            else:
                                fl = deequ_flag(src, ref, dom)
                        except Exception as e:
                            fl = None  # record adapter error, don't crash the grid
                        if fl is None:
                            continue
                        if is_deg:
                            per[bname]["deg"] += 1; per[bname]["tp"] += int(fl)
                        else:
                            per[bname]["cl"] += 1; per[bname]["cf"] += int(fl)
                for bname, c in per.items():
                    rows.append({
                        "domain": domain_name, "mode": mode, "severity": severity,
                        "seed": seed, "baseline": bname,
                        "det_recall": round(c["tp"]/c["deg"], 4) if c["deg"] else "",
                        "false_flag_rate": round(c["cf"]/c["cl"], 4) if c["cl"] else "",
                    })
bdf = pd.DataFrame(rows)
out = pathlib.Path(OUT, "baselines_detection.csv")   # OUT is writable (Kaggle: /kaggle/working)
with open(out, "w", newline="") as f:
    f.write("# External baselines (GE/TFDV/Deequ) default checks on the gate's sources\n")
    f.write("# generated by experiments/baselines/baselines_colab.ipynb (Colab/Kaggle)\n")
    f.write(f"# domains={DOMAINS} severities={SEVERITIES} seeds={len(SEEDS)} "
            f"n_sources={N_SOURCES} n_degraded={N_DEGRADED} data_root={DATA_ROOT}\n")
    bdf.to_csv(f, index=False)
print("wrote", out, len(bdf), "rows")
bdf.groupby(["baseline", "mode"])["det_recall"].apply(
    lambda s: pd.to_numeric(s, errors="coerce").mean()).unstack(0)

In [ ]:
# 8. Compare to the gate. Load the committed crossdomain CSVs (gate det_recall at
#    epsilon=inf) and place them beside the baselines, per domain x mode.
def load_gate():
    frames = []
    for d in DOMAINS:
        p = pathlib.Path(REPO_DIR, "experiments", "results", f"crossdomain_{d}.csv")
        if not p.exists():
            continue
        g = pd.read_csv(p, comment="#")
        g = g[(g["mode"] != "none") & (g["epsilon"].astype(str) == "inf")]
        g = g.groupby(["domain", "mode"])["det_recall"].mean().reset_index()
        g["baseline"] = "gate"; frames.append(g)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

gate = load_gate()
base = bdf[bdf["mode"] != "none"].copy()
base["det_recall"] = pd.to_numeric(base["det_recall"], errors="coerce")
base = base.groupby(["domain", "mode", "baseline"])["det_recall"].mean().reset_index()
compare = pd.concat([gate, base], ignore_index=True)
compare.pivot_table(index=["domain", "mode"], columns="baseline", values="det_recall")

## Reading the result

The expected story (to confirm on the real run):

- **structural / completeness** — all three baselines and the gate should detect these; they are exactly what off-the-shelf validators are built for. The gate should *not* look novel here, and that is fine.
- **distributional** — GE catches it only via explicit `between` bounds; TFDV only with the drift comparator enabled; Deequ only via the mean-band. The gate's cross-source consensus divergence is the principled version.
- **label** — the interesting column. A symmetric flip on a balanced target moves no marginal, so GE's mean-band, TFDV, and Deequ all miss it — the same blind spot the gate's marginal detector has, and precisely what the **confident-learning detector** (`src/domains/label_quality.py`, Finding D) is for. That contrast is the paper's point: default validators and a marginal gate both miss symmetric label noise; the cross-validated self-disagreement statistic does not.

**Then:** copy `baselines_detection.csv` back into the repo, and report per-mode `gate` vs `great_expectations`/`tfdv`/`deequ` `det_recall`. The takeaway to support: baselines match the gate on structural/completeness (no raw-check novelty claimed) and fall behind on distributional/label — where the DP-private, cross-source, decision layer earns its keep.